# Streaming with LangSmith API

## Steps: 
1. Have Agent.py file ready in the studio folder
2. keep .env file with LangSmith API keys and GOOGle API keys ready
3. set langgraph.json file ready
4. To start the local development server,
   run the following command in your terminal in the /studio directory in this module: BAsh
   ' langgraph dev '

### check for output:
 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs

This in-memory server is designed for development and testing.
For production use, please use LangSmith Deployment.

6. Copy url = http://127.0.0.1:2024

#### Note: 
- check for more info : https://docs.langchain.com/langsmith/quick-start-studio#local-development-server
- check for LangSmith setup guide in the begining of the course.

7. ### Connecting to the Local Server Using the SDK
You use the LangGraph SDK to talk to your local agent: python
```
from langgraph_sdk import get_client

url = "http://127.0.0.1:2024"
client = get_client(url=url)
```
This client object lets you:

    - list assistants
    - create threads
    - run your agent
    - stream results

8. ### Create a thread for storing event checkpoints?
A thread is a conversation session.

It stores:

    - messages
    - memory
    - state
    - checkpoints

You create one like this: python
```
thread = await client.threads.create()
```
This returns: python
```
{"thread_id": "abc123"}
```
You will use this thread ID for the entire conversation.

9.  ### Running the Agent With breakpoints
note: learn more here[API breakpoints](https://docs.langchain.com/langsmith/add-human-in-the-loop)

You run your agent like this: python
```
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="agent",
    input=initial_input,
    stream_mode="values",
    interrupt_before=["tools"],
):
```
Note:  We can add interrupt_before=["node"] when compiling the graph that is running in Studio or with the API, you can also pass interrupt_before to the stream method directly.

#### What happens?
Your agent runs step by step:

    - LLM node
    - breakpoint to check with user
    - Tool node
    - LLM node
    - Final answer
After each step, LangGraph sends you a chunk containing the updated state.

In [1]:
from langgraph_sdk import get_client

url = "http://127.0.0.1:2024"
client  = get_client(url=url)

# Search all hosted graphs
assistants = await client.assistants.search()
# list the agents
assistants

[{'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca',
  'graph_id': 'agent',
  'config': {},
  'context': {},
  'metadata': {'created_by': 'system'},
  'name': 'agent',
  'created_at': '2026-04-29T20:22:12.625636+00:00',
  'updated_at': '2026-04-29T20:22:12.625636+00:00',
  'version': 1,
  'description': None}]

In [12]:
# create a thread
thread = await client.threads.create()
print(thread)

{'thread_id': '019ddb36-4bf7-7490-ad29-387c0cb16866', 'created_at': '2026-04-29T21:47:41.940139+00:00', 'updated_at': '2026-04-29T21:47:41.940139+00:00', 'state_updated_at': '2026-04-29T21:47:41.940139+00:00', 'metadata': {}, 'status': 'idle', 'config': {}, 'values': None}


In [13]:
# stream events
from langchain_core.messages import HumanMessage
msg_input = [ HumanMessage(content="Divide 30 and 10")]
async for event in client.runs.stream(
        thread["thread_id"],   # which conversation
        "agent",               # which assistant
        input={"messages": msg_input},       # what the user said
        stream_mode="values",  # stream full state after each step
        interrupt_before=["tools"], # add breakpoint before tool node
    ):
    #print(event)
    #capture messages
    messages = event.data.get('messages',[])
    # if messages exist
    if messages:
        print("\n", messages[-1])
    print("-" * 50)

--------------------------------------------------

 {'content': 'Divide 30 and 10', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'human', 'name': None, 'id': 'd676cb96-293e-45bd-8b69-b118ebe5934c'}
--------------------------------------------------

 {'content': [], 'additional_kwargs': {'function_call': {'name': 'division', 'arguments': '{"b": 10, "a": 30}'}, '__gemini_function_call_thought_signatures__': {'d53bfdf3-7fcb-404b-9252-fcd4fcacabec': 'EjQKMgEMOdbH+3bcjLiD0leFiV2KvRj0O3k5W+GH66s7droQtSiMxHPbetD9OxBQ2eo7rbAl'}}, 'response_metadata': {'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, 'type': 'ai', 'name': None, 'id': 'lc_run--019ddb36-528c-7670-9468-74877f2231f7-0', 'tool_calls': [{'name': 'division', 'args': {'b': 10, 'a': 30}, 'id': 'd53bfdf3-7fcb-404b-9252-fcd4fcacabec', 'type': 'tool_call'}], 'invalid_tool_calls': [], 'usage_metadata': {'input_tokens': 254, 'output_tokens': 18, 't

In [14]:
toolcall = messages[-1]['tool_calls']
toolcall

[{'name': 'division',
  'args': {'b': 10, 'a': 30},
  'id': 'd53bfdf3-7fcb-404b-9252-fcd4fcacabec',
  'type': 'tool_call'}]

### Graph is interruped by the breakpointbefore the tool node

Ask  user to continue to use the tool if tool_Calls exist

In [15]:
# user approval
if toolcall:
    user_approval = input("Do you want to continue?(yes/no): ").lower()
    if user_approval == 'yes':
        # pass None as input to the graph and continue from the last statenusing the thread_ID
        async for event in client.runs.stream(
            thread["thread_id"],
            "agent",
            input=None,
            stream_mode="values",
            interrupt_before=["tools"],
        ):
            print(f"Receiving new event of type: {event.event}...")
            messages = event.data.get('messages', [])
            if messages:
                print(messages[-1])
            print("-" * 50)
        



Do you want to continue?(yes/no):  yes


Receiving new event of type: metadata...
--------------------------------------------------
Receiving new event of type: values...
{'content': [], 'additional_kwargs': {'function_call': {'name': 'division', 'arguments': '{"b": 10, "a": 30}'}, '__gemini_function_call_thought_signatures__': {'d53bfdf3-7fcb-404b-9252-fcd4fcacabec': 'EjQKMgEMOdbH+3bcjLiD0leFiV2KvRj0O3k5W+GH66s7droQtSiMxHPbetD9OxBQ2eo7rbAl'}}, 'response_metadata': {'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, 'type': 'ai', 'name': None, 'id': 'lc_run--019ddb36-528c-7670-9468-74877f2231f7-0', 'tool_calls': [{'name': 'division', 'args': {'b': 10, 'a': 30}, 'id': 'd53bfdf3-7fcb-404b-9252-fcd4fcacabec', 'type': 'tool_call'}], 'invalid_tool_calls': [], 'usage_metadata': {'input_tokens': 254, 'output_tokens': 18, 'total_tokens': 272, 'input_token_details': {'cache_read': 0}}}
--------------------------------------------------
Receiving new event o